In [10]:
%pip install -q ultralytics mlflow boto3

Note: you may need to restart the kernel to use updated packages.


In [11]:
import random
import shutil
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import torch
import yaml
from ultralytics import YOLO


@dataclass
class TrainConfig:
    """Paramètres de l'entraînement."""
    zip_path: Path = Path("dataset-yolo-aquateam.zip")
    data_dir: Path = Path("datasets/flexistaff")
    model: str = "yolo11s.pt"
    epochs: int = 100
    imgsz: int = 640
    batch: int = 4
    device: str = "auto"
    workers: int = 2
    cpu_threads: int = 2
    cpu_batch: int = 2

    def resolve_device(self) -> str:
        """Retourne '0' si CUDA est disponible, sinon 'cpu'."""
        if self.device == "cpu":
            return "cpu"
        if self.device == "auto":
            return "0" if torch.cuda.is_available() else "cpu"
        if not torch.cuda.is_available():
            raise RuntimeError(f"GPU demandé ({self.device}), mais CUDA est indisponible")
        return self.device

    def training_options(self) -> dict:
        device = self.resolve_device()
        if device == "cpu":
            torch.set_num_threads(self.cpu_threads)
            cv2.setNumThreads(self.cpu_threads)
            print("CPU : threads limités, batch réduit, workers désactivés ; la RAM n'est pas plafonnée.")
            return {"device": device, "batch": min(self.batch, self.cpu_batch), "workers": 0, "cache": False}
        print(f"GPU sélectionné : {device} | CUDA PyTorch : {torch.version.cuda}")
        return {"device": device, "batch": self.batch, "workers": self.workers, "cache": False}


class DatasetPreparer:
    """Dézippe le dataset et vérifie sa structure."""

    def __init__(self, config: TrainConfig):
        self.config = config

    def extract(self) -> Path:
        """Dézippe l'archive si le dossier cible n'existe pas encore."""
        target = self.config.data_dir
        if not target.exists():
            with zipfile.ZipFile(self.config.zip_path) as zf:
                zf.extractall(target)
        return target

    def find_data_yaml(self) -> Path:
        """Retourne le chemin du data.yaml, où qu'il soit dans l'archive."""
        found = list(self.config.data_dir.rglob("data.yaml"))
        if not found:
            raise FileNotFoundError("data.yaml introuvable dans le dataset")
        return found[0]


class DatasetSplitter:
    """Découpe train.txt en train/val et met à jour data.yaml."""

    def __init__(self, data_yaml: Path, val_ratio: float = 0.2,
                 seed: int = 42, by_folder: bool = True):
        self.data_yaml = data_yaml
        self.root = data_yaml.parent
        self.val_ratio = val_ratio
        self.seed = seed
        self.by_folder = by_folder

    def _resolve(self, line: str) -> Path:
        """Résout un chemin de train.txt, avec ou sans le préfixe 'data/'."""
        path = self.root / line.strip()
        if not path.exists() and line.startswith("data/"):
            path = self.root / line[len("data/"):]
        if not path.exists():
            raise FileNotFoundError(f"Image introuvable : {line}")
        return path.resolve()

    def read_images(self) -> list[str]:
        """Lit train_all.txt et retourne les chemins d'images en absolu."""
        lines = (self.root / "train_all.txt").read_text().splitlines()
        return [str(self._resolve(line)) for line in lines if line.strip()]

    def split(self, images: list[str]) -> tuple[list[str], list[str]]:
        """Sépare train / val, par dossier (lot) ou image par image."""
        groups = defaultdict(list)
        for img in images:
            groups[str(Path(img).parent) if self.by_folder else img].append(img)
        keys = sorted(groups)
        random.Random(self.seed).shuffle(keys)
        val = []
        for key in keys:
            if len(val) >= len(images) * self.val_ratio:
                break
            val += groups[key]
        val_set = set(val)
        return [i for i in images if i not in val_set], val

    def write_lists(self, train: list[str], val: list[str]) -> None:
        """Écrit train.txt et val.txt avec des chemins absolus."""
        (self.root / "train.txt").write_text("\n".join(train))
        (self.root / "val.txt").write_text("\n".join(val))

    def update_yaml(self) -> None:
        """Pointe data.yaml vers train.txt et val.txt."""
        config = yaml.safe_load(self.data_yaml.read_text())
        config.update(path=str(self.root.resolve()), train="train.txt", val="val.txt")
        self.data_yaml.write_text(yaml.safe_dump(config, allow_unicode=True))

    def run(self) -> None:
        """Exécute le split une seule fois (ne fait rien si val.txt existe)."""
        if (self.root / "val.txt").exists():
            return
        backup = self.root / "train_all.txt"
        if not backup.exists():
            shutil.copy(self.root / "train.txt", backup)
        train, val = self.split(self.read_images())
        self.write_lists(train, val)
        self.update_yaml()


class YoloTrainer:
    """Entraîne un YOLO à partir d'une TrainConfig."""

    def __init__(self, config: TrainConfig, data_yaml: Path):
        self.config = config
        self.data_yaml = data_yaml

    def train(self):
        """Lance l'entraînement et retourne les résultats."""
        options = self.config.training_options()
        model = YOLO(self.config.model)
        return model.train(
            data=str(self.data_yaml),
            epochs=self.config.epochs,
            imgsz=self.config.imgsz,
            **options,
            fliplr=0.0,
            name="flexistaff",
        )

In [12]:
archive_path = Path("zip/aqua-team.zip")
if not archive_path.is_file():
    archive_path = Path("zip/aqua-team.zip")
if not archive_path.is_file():
    raise FileNotFoundError("aqua-team.zip introuvable dans le dossier courant ou ai_training/")

config = TrainConfig(
    zip_path=archive_path,
    data_dir=archive_path.parent / "datasets" / "flexistaff",
)
print("PyTorch :", torch.__version__, "| CUDA compilé :", torch.version.cuda)
print("CUDA :", torch.cuda.is_available(), "| device retenu :", config.resolve_device())
if torch.cuda.is_available():
    print("GPU 0 :", torch.cuda.get_device_name(0))
else:
    print("GPU indisponible : vérifier le pilote NVIDIA, la version CUDA de PyTorch et CUDA_VISIBLE_DEVICES.")

PyTorch : 2.6.0+cu124 | CUDA compilé : 12.4
CUDA : True | device retenu : 0
GPU 0 : NVIDIA GeForce RTX 3050 Laptop GPU


In [13]:
preparer = DatasetPreparer(config)
preparer.extract()
data_yaml = preparer.find_data_yaml()
data_yaml

PosixPath('zip/datasets/flexistaff/data.yaml')

In [14]:
DatasetSplitter(data_yaml, val_ratio=0.2, by_folder=False).run()
print(
    "Images train :",
    len((data_yaml.parent / "train.txt").read_text().splitlines()),
    "| validation :",
    len((data_yaml.parent / "val.txt").read_text().splitlines()),
)

Images train : 29 | validation : 8


In [15]:
import json
import os

import boto3
import mlflow
from IPython.display import Markdown, display


credential_files = sorted(Path.cwd().glob("identifiants-*.json"))
if not credential_files:
    credential_files = sorted(Path.cwd().parent.glob("identifiants-*.json"))
if len(credential_files) != 1:
    raise FileNotFoundError("Placer un seul fichier identifiants-<login>.json à la racine du dépôt.")

credentials = json.loads(credential_files[0].read_text())
mlflow_credentials = credentials["mlflow"]
s3_credentials = credentials["s3"]
team = credentials["team"]

os.environ["MLFLOW_TRACKING_URI"] = mlflow_credentials["tracking_uri"]
os.environ["MLFLOW_TRACKING_USERNAME"] = mlflow_credentials["username"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = mlflow_credentials["password"]
os.environ["MLFLOW_S3_ENDPOINT_URL"] = s3_credentials["endpoint_url"]
os.environ["AWS_ACCESS_KEY_ID"] = s3_credentials["access_key_id"]
os.environ["AWS_SECRET_ACCESS_KEY"] = s3_credentials["secret_access_key"]
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

mlflow.set_tracking_uri(mlflow_credentials["tracking_uri"])
experiment_name = f"flexistaff_{team}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = experiment_name
mlflow.set_experiment(experiment_name)

run_name = f"{team}_flexistaff"
os.environ["MLFLOW_RUN"] = run_name
with mlflow.start_run(run_name=run_name) as mlflow_run:
    mlflow.set_tags({"team": team, "model": config.model})
    mlflow.log_params({"model": config.model, "epochs": config.epochs, "imgsz": config.imgsz})
    results = YoloTrainer(config, data_yaml).train()

    run_dir = Path(results.save_dir)
    best_model = run_dir / "weights" / "best.pt"
    if not best_model.is_file():
        raise FileNotFoundError(f"Poids entraînés introuvables : {best_model}")

    mlflow.log_metrics({
        "mAP50": results.box.map50,
        "precision": results.box.mp,
        "recall": results.box.mr,
    })
    mlflow.log_artifact(str(best_model), artifact_path="checkpoint")

    s3_client = boto3.client(
        "s3",
        endpoint_url=s3_credentials["endpoint_url"],
        aws_access_key_id=s3_credentials["access_key_id"],
        aws_secret_access_key=s3_credentials["secret_access_key"],
        region_name="us-east-1",
    )
    prefix_parts = [s3_credentials["prefix"].strip("/"), mlflow_run.info.run_id]
    object_prefix = "/".join(part for part in prefix_parts if part)
    uploaded_files = 0
    for artifact in sorted(run_dir.rglob("*")):
        if artifact.is_file():
            relative_path = artifact.relative_to(run_dir).as_posix()
            object_key = f"{object_prefix}/{relative_path}" if object_prefix else relative_path
            s3_client.upload_file(str(artifact), s3_credentials["bucket"], object_key)
            uploaded_files += 1

print("Résultats dans :", run_dir)
print("Artefacts envoyés vers RustFS :", uploaded_files)
display(Markdown(
    "**Services de suivi** : "
    "[MLflow](http://13.37.102.106:5000) · "
    "[RustFS](http://13.37.102.106:9000)"
))

GPU sélectionné : 0 | CUDA PyTorch : 12.4
Ultralytics 8.4.215 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=zip/datasets/flexistaff/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale

2026/11/17 11:31:11 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/11/17 11:31:11 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


MLflow: logging run_id(5586fa1699ef4a66a21b2075bb57d85c) to http://13.37.102.106:5000
MLflow: disable with 'yolo settings mlflow=False'
Using 29 train, 8 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /home/tangilc/iadev/flexistaff/ai_training/runs/detect/flexistaff-9
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      1.63G      2.713      4.259      2.017         44        640: 100% ━━━━━━━━━━━━ 8/8 1.6it/s 5.1s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all          8        516       0.83      0.146       0.15      0.111

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      1.78G      2.773        4.3      2.047         22        640: 100% ━━━━━━━━━━━━ 8/8 3.4it/s 2.3s0.3s
                 Class     Images  Instances   

AttributeError: 'NoneType' object has no attribute 'strip'